In [65]:
import numpy as np 
import tensorflow as tf
# from tensorflow.keras import layers, models
# from tensorflow.keras.applications import ResNet50
from keras.activations import relu, softmax, leaky_relu, sigmoid, tanh
from keras.optimizers import Adam, SGD, RMSprop, Adagrad
from keras.layers import Dense, RNN, LSTM, GRU, Dropout, SimpleRNN, Embedding, TextVectorization, CategoryEncoding
from keras.utils import pad_sequences
from keras.preprocessing import sequence
from keras.datasets import imdb
from keras.models import Sequential
from keras.losses import SparseCategoricalCrossentropy
from keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
# from tensorflow.keras.preprocessing import one_hot
from keras.models import load_model
import re


In [66]:
model = load_model("rnn_imdb_model.h5")
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ (32, 500, 128)         │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn_2 (SimpleRNN)        │ (32, 128)              │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (32, 1)                │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,313,027 (5.01 MB)

 Trainable params: 1,313,025 (5.01 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)

In [67]:
word_index = imdb.get_word_index()
def encode_review(text):
    text = text.lower()
    

In [68]:
word_index = imdb.get_word_index()
reverse_word_index = {value: key for key, value in word_index.items()}

def encode_review(text):
    return [word_index.get(word, 2) + 3 for word in text.lower().split()]
    
def decode_review(text):
    # Preprocess the input text
    maxlen = 500
    # vectorizer = TextVectorization(max_tokens=20000, output_mode='int')
    # vectorizer.adapt([text])
    input_seq = encode_review(text) # vectorizer([text])
    # print(f"Input sequence: {input_seq}")
    input_seq = pad_sequences([input_seq], maxlen=maxlen, padding='pre')
    # print(input_seq)

    # Make prediction
    prediction = model.predict(input_seq)
    sentiment = "Positive" if prediction[0][0] >= 0.5 else "Negative"
    return sentiment, prediction[0][0]


def decode_review(encoded_review): # Numbers to text
    return ' '.join([reverse_word_index.get(i - 3, '?') for i in encoded_review])

# Function to preprocess user input
def preprocess_text(text):
    words = text.lower().split()
    encoded_review = [word_index.get(word, 2) + 3 for word in words]
    padded_review = sequence.pad_sequences([encoded_review], maxlen=500)
    return padded_review

In [69]:
def predict_sentiment(review):
    preprocessed_input=preprocess_text(review)

    prediction=model.predict(preprocessed_input)

    sentiment = 'Positive' if prediction[0][0] > 0.5 else 'Negative'
    
    return sentiment, prediction[0][0]

In [70]:
review = "It was great"
sentiment, confidence = predict_sentiment(review)
# # print(f"Review: {review}")
# # intReview = encode_review(review)
# # input_seq = pad_sequences([intReview], maxlen=500, padding='pre')
print(f"Sentiment: {sentiment}, Confidence: {confidence}")
# # print(input_seq)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 128ms/step
Sentiment: Positive, Confidence: 0.9998784065246582


In [ ]:
# 